# 03. SQL 적재

- **목적**: 전처리한 테이블 3개를 PostgreSQL(`cafe_location_project`)에 적재하고 기본 키 설정
- **데이터 출처**
  - `data/processed/커피_매출.csv`, `커피_점포.csv`, `영역.csv` (`02_전처리` 출력, 2021년 1분기 ~ 2026년 2분기)
  - `sql/01_기본키.sql`
- **출력**: DB 테이블 `커피_매출`, `커피_점포`, `영역`
- **사전 준비**: 프로젝트 루트 `.env`에 `PGHOST`, `PGPORT`, `PGUSER`, `PGPASSWORD`, `PGDATABASE` 기입 (비밀번호는 코드에 두지 않음)
- **작성일**: 2026-10-16 (출력은 2026-10-15 실행 결과)

## 결론 요약
- 커피_매출 23,700행, 커피_점포 33,263행, 영역 1,650행 적재, 파일과 DB 행 수 모두 일치
- 기본 키 설정 확인: `영역`(상권_코드), `커피_매출`·`커피_점포`(기준_년분기_코드, 상권_코드)

## 설정

In [1]:
import sys
from pathlib import Path

import pandas as pd
from sqlalchemy import text

sys.path.append("..")  # notebooks/에서 실행하므로 프로젝트 루트의 src를 import 경로에 추가
from src.data_io import CODE_DTYPES
from src.db import get_engine

pd.set_option("display.max_columns", 20)
pd.set_option("display.max_rows", 60)

In [2]:
ENV_PATH = Path("../.env")
PROCESSED_DIR = Path("../data/processed")
PK_SQL_PATH = Path("../sql/01_기본키.sql")

TABLES = ["커피_매출", "커피_점포", "영역"]  # csv 파일명 = 테이블명
CHUNK_SIZE = 5000  # 한 번에 INSERT하는 행 수, 메모리·속도 절충

## DB 연결

- 접속 정보 오류를 적재 전에 잡기 위해 현재 DB 이름부터 확인

In [3]:
engine = get_engine(ENV_PATH)

with engine.connect() as conn:
    print(conn.execute(text("SELECT current_database()")).scalar())

cafe_location_project


## 데이터 로드와 적재

- `if_exists="replace"`로 매번 테이블을 새로 생성 → 노트북을 다시 실행해도 결과가 같음
- 적재 직후 DB 행 수를 파일 행 수와 비교

In [4]:
for table in TABLES:
    table_df = pd.read_csv(PROCESSED_DIR / f"{table}.csv", dtype=CODE_DTYPES)
    table_df.to_sql(table, engine, if_exists="replace", index=False, chunksize=CHUNK_SIZE)

    with engine.connect() as conn:
        n_loaded = conn.execute(text(f'SELECT COUNT(*) FROM "{table}"')).scalar()
    print(table, "| 파일", len(table_df), "| 적재", n_loaded, "| 일치", len(table_df) == n_loaded)
    assert len(table_df) == n_loaded

커피_매출 | 파일 23700 | 적재 23700 | 일치 True


커피_점포 | 파일 33263 | 적재 33263 | 일치 True
영역 | 파일 1650 | 적재 1650 | 일치 True


## 기본 키 설정

- `replace`로 테이블을 다시 만들면 기본 키가 사라지므로 적재할 때마다 `sql/01_기본키.sql` 재실행
- 기본 키는 중복 행 적재를 DB 단에서 막는 장치

In [5]:
pk_statements = [s for s in PK_SQL_PATH.read_text(encoding="utf-8").split(";") if s.strip()]

with engine.begin() as conn:  # 문장 하나라도 실패하면 전체 롤백
    for statement in pk_statements:
        conn.execute(text(statement))

In [6]:
pk_check_sql = """
SELECT tc.table_name AS 표, string_agg(kcu.column_name, ', ' ORDER BY kcu.ordinal_position) AS 기본_키
FROM information_schema.table_constraints tc
JOIN information_schema.key_column_usage kcu ON tc.constraint_name = kcu.constraint_name
WHERE tc.constraint_type = 'PRIMARY KEY' AND tc.table_schema = 'public'
GROUP BY tc.table_name
"""
pd.read_sql(pk_check_sql, engine)

,표,기본_키
0,영역,상권_코드
1,커피_매출,"기준_년분기_코드, 상권_코드"
2,커피_점포,"기준_년분기_코드, 상권_코드"


### 관찰
- 테이블 3개 모두 기본 키 설정 확인
- `영역`은 상권_코드, `커피_매출`·`커피_점포`는 (기준_년분기_코드, 상권_코드)

## 결론과 다음 단계

- 테이블 3개 적재, 파일과 DB 행 수 일치, 기본 키 설정 완료
- 다음 단계: `04_추천.ipynb`에서 `sql/02_추천지표.sql`로 지표 계산 후 후보 상권 추천